# Sigma Prompting Challenge Notebook

Create Sigma detection rules from a threat intel excerpt and compare prompting strategies.

Notebook recovery note: this file was re-saved to resolve a transient load issue.

## 1) Setup (Model + Inference)

You can do this challenge in three ways:
- In this notebook 
- In ChatGPT
- In Ollama

however you like to do it. Feel free to experiment with differnt models and methods.

In [7]:
# If needed, uncomment:
# %pip install -q transformers torch sentencepiece huggingface_hub

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'google/gemma-3-4b-it'  # change if needed, maybe with a quantized version

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_text(prompt, max_new_tokens=450, temperature=0.3, top_k=50, top_p=0.95, repetition_penalty=1.05, do_sample=True):
    inputs = tokenizer(prompt, return_tensors='pt')
    input_ids = inputs['input_ids']
    attention_mask = inputs.get('attention_mask')

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print('Model loaded:', MODEL_ID)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: google/gemma-3-4b-it


## 2) Challenge Context (Threat Intel Excerpt)

Task: Create Sigma rules from this report excerpt.

In [8]:
THREAT_EXCERPT = (
    "Persistence Mechanism\n\n"
    "Malicious execution begins by setting up persistence, but only if the process's current "
    "working directory is not C:\\Windows\\System32. This check prevents persistence from being "
    "established when executed via tools like rundll32.exe, though the malware is still executed. "
    "If persistence is required, GRAPELOADER:\n\n"
    "- Copies the content of the delivered archive wine (2).zip to "
    "C:\\Users\\User\\AppData\\Local\\POWERPNT\\.\n"
    "- Creates a Run registry key at SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run with "
    "the entry POWERPNT, pointing to C:\\Users\\User\\AppData\\Local\\POWERPNT\\wine.exe."
)

print(THREAT_EXCERPT)

Persistence Mechanism

Malicious execution begins by setting up persistence, but only if the process's current working directory is not C:\Windows\System32. This check prevents persistence from being established when executed via tools like rundll32.exe, though the malware is still executed. If persistence is required, GRAPELOADER:

- Copies the content of the delivered archive wine (2).zip to C:\Users\User\AppData\Local\POWERPNT\.
- Creates a Run registry key at SOFTWARE\Microsoft\Windows\CurrentVersion\Run with the entry POWERPNT, pointing to C:\Users\User\AppData\Local\POWERPNT\wine.exe.


## 3) Section A: Basic Prompt (Baseline)

Run the minimal prompt first:
- `Create a sigma detection rule.`

Then compare this result to your stronger prompts later.

Feel free to configure the parameters as you like and experiment with them.

In [9]:
basic_prompt = f'''Create a sigma detection rule.\n\nThreat report excerpt:\n{THREAT_EXCERPT}'''

basic_output = generate_text(
    basic_prompt,
    max_new_tokens=420,
    temperature=0.3,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.05,
    do_sample=True
)

print('=== Basic Prompt ===')
print(basic_prompt)
print('\n=== Model Output ===')
print(basic_output)

=== Basic Prompt ===
Create a sigma detection rule.

Threat report excerpt:
Persistence Mechanism

Malicious execution begins by setting up persistence, but only if the process's current working directory is not C:\Windows\System32. This check prevents persistence from being established when executed via tools like rundll32.exe, though the malware is still executed. If persistence is required, GRAPELOADER:

- Copies the content of the delivered archive wine (2).zip to C:\Users\User\AppData\Local\POWERPNT\.
- Creates a Run registry key at SOFTWARE\Microsoft\Windows\CurrentVersion\Run with the entry POWERPNT, pointing to C:\Users\User\AppData\Local\POWERPNT\wine.exe.

=== Model Output ===
Create a sigma detection rule.

Threat report excerpt:
Persistence Mechanism

Malicious execution begins by setting up persistence, but only if the process's current working directory is not C:\Windows\System32. This check prevents persistence from being established when executed via tools like rundll32

### Reflection Questions (Section A)
1. Is the output valid Sigma YAML?
2. Does it include the right log source and detection logic?
3. What key details from the excerpt are missing?

## 4) Section B: TCREI Prompting (Google Framework)

Build your own stronger prompt using TCREI:
- Task (Persona + Format)
- Context
- References
- Evaluate
- Iterate

You should craft this prompt yourselves and compare against Section A.

Note: For the evaluate and iterate step, it might be better to use an interactive Chat-Bot such as Ollama or ChatGPT here.

In [10]:
# STUDENT TASK: Replace this with your own TCREI prompt
tcrei_prompt = f""""""

tcrei_output = generate_text(
    tcrei_prompt,
    max_new_tokens=520,
    temperature=0.3,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.05,
    do_sample=True
)

print('=== TCREI Prompt ===')
print(tcrei_prompt)
print('\n=== Model Output ===')
print(tcrei_output)

=== TCREI Prompt ===


=== Model Output ===
The 2023-24 NBA season is in full swing, and the league is brimming with exciting storylines, star performances, and unexpected developments. Here's a breakdown of the top narratives shaping the season so far:

**1. The Nuggets' Reign Continues (For Now):**

* **The Story:** The Denver Nuggets are dominating the Western Conference, largely thanks to Nikola Jokic's continued brilliance and Jamal Murray's resurgence. They've built an impressive record and look like the team to beat.
* **Key Factors:** Jokic's MVP-caliber play, Murray's improved health and shooting, strong defense, and coaching from Michael Malone.
* **What to Watch For:** Can they maintain this level of dominance throughout the playoffs? How will they handle potential challenges from the Lakers or Suns?


**2. Lakers' Rebuild in Motion (and a Little Chaotic):**

* **The Story:**  The Lakers are undergoing a significant rebuild, spearheaded by LeBron James and Anthony Davis. Whi

### Reflection Questions (Section B)
1. Did TCREI improve structure or detection quality vs basic prompt?
2. Which TCREI part had the biggest impact?
3. What would you change in your TCREI prompt for another iteration?

## 5) Section C: Zero-shot vs One-shot vs Few-shot

In this section, implement a system prompt using:
- Persona
- Context and scenario details
- Conversation type
- Output Format

Implementation pattern for all runs:
- **System prompt** = agent behavior definition (PCCSR)
- **Instruction prompt** = `#Input` with CTI excerpt + rule task
- **Examples** = optional `#Input` / `#Output` pairs (for one-shot and few-shot)

Recommended places:
- https://sigmahq.io/
- https://github.com/SigmaHQ/sigma/tree/master/rules

You can do this in notebook, ChatGPT, or Ollama. Cells below let you run directly here.

In [ ]:
# Prompt building blocks

# STUDENT TASK: Fill this with your PCCSR system prompt. Keep empty at first.
system_prompt = ''

# Predefined instruction prompt based on CTI input
instruction_prompt = f'''
#Input
{THREAT_EXCERPT}

#Output
'''

# STUDENT TASK: Populate examples from SigmaHQ (keep template format)
example_1 = '''
#Input 
<student fill this out>
#Output 
<student fill this out>'''

example_2 = '''
#Input 
<student fill this out>
#Output 
<student fill this out>'''

example_3 = '''
#Input 
<student fill this out>
#Output 
<student fill this out>'''

def build_prompt(system_prompt, instruction_prompt, examples=None):
    examples = examples or []
    parts = []
    if system_prompt.strip():
        parts.append('### System\n' + system_prompt.strip())
    if examples:
        parts.append('### Examples\n' + '\n\n'.join([e.strip() for e in examples if e.strip()]))
    parts.append('### Instruction\n' + instruction_prompt.strip())
    return '\n\n'.join(parts)

# Zero-shot = system + instruction (no examples)
zero_shot_prompt = build_prompt(system_prompt, instruction_prompt, examples=[])
zero_shot_output = generate_text(zero_shot_prompt, max_new_tokens=520, temperature=0.3)
print('=== Zero-shot Prompt ===')
print(zero_shot_prompt)
print('=== Zero-shot Output ===')
print(zero_shot_output)

In [ ]:
# One-shot = system + 1 example + instruction
one_shot_prompt = build_prompt(system_prompt, instruction_prompt, examples=[example_1])
one_shot_output = generate_text(one_shot_prompt, max_new_tokens=620, temperature=0.3)
print('=== One-shot Prompt ===')
print(one_shot_prompt)
print('=== One-shot Output ===')
print(one_shot_output)

In [ ]:
# Few-shot = system + 3 examples + instruction
few_shot_prompt = build_prompt(system_prompt, instruction_prompt, examples=[example_1, example_2, example_3])
few_shot_output = generate_text(few_shot_prompt, max_new_tokens=700, temperature=0.3)
print('=== Few-shot Prompt ===')
print(few_shot_prompt)
print('=== Few-shot Output ===')
print(few_shot_output)

### Reflection Questions (Section C)
1. Did you observe differences in rule quality between zero-shot, one-shot, and few-shot?
2. Which setup produced the most realistic and operational rule?
3. Did examples improve ATT&CK mapping, fields, and detection condition quality?
4. What happened when you changed the example quality (good vs weak examples)?

## 6) Final Comparison Against Manual Baseline Rule

Compare your generated outputs against this manually crafted baseline:

```yaml
title: Suspicious Run Key from AppData
id: E6751CA0-F8A3-48FB-9C1C-29D22F476CC6
status: experimental
description: Detects the suspicious RUN keys created by software located in AppData
references:
    - https://research.checkpoint.com/2025/apt29-phishing-campaign/
author: Jonas Kaspereit
date: 2025-11-25
tags:
    - attack.persistence
    - attack.T1547.001
logsource:
    category: registry_event
    product: windows
detection:
    selection:
        Image|contains:
            - '\Downloads\'
        EventID:
        - 13
        TargetObject|contains:
            - '\Software\Microsoft\Windows\CurrentVersion\Run'
            - '\Software\Microsoft\Windows\CurrentVersion\RunOnce'
    condition: selection
falsepositives:
    - Software installers downloaded and used by users
level: medium
```

### Final Reflection
1. Which prompting strategy came closest to the manual baseline quality?
2. Where did LLM outputs still fail compared to expert-crafted detection logic?
3. What prompt improvements would you apply in a real SOC workflow?
4. If using ChatGPT or Ollama, were results better/worse than the notebook model? Why?